# 11 · Limpieza: Informe por segmento Elite y Premium_anonimizado

Reporte de Zoho CRM (13 de noviembre de 2026) con las empresas afiliadas de los segmentos **Elite** y **Premium**, su KAM y cuándo vence su afiliación. Una fila = una empresa.

- Entrada: `datos/originales/Informe por segmento Elite y Premium_anonimizado.xlsx` (hoja `Sheet0`)
- Salida: `datos/limpios/11_segmento_elite_premium_limpio.csv`

In [ ]:
from pathlib import Path
import re

import pandas as pd

BASE = Path.cwd().parent
ORIGINALES = BASE / "datos" / "originales"
LIMPIOS = BASE / "datos" / "limpios"
LIMPIOS.mkdir(exist_ok=True)

## 1. Cómo viene un reporte agrupado de Zoho
Es el mismo formato del cuaderno 07_08. Las filas 0 a 5 son títulos (con `Cantidad de registros : n`) y la fila 6 tiene los encabezados.
Los datos vienen **agrupados**: cada etiqueta de grupo aparece solo en la primera fila del grupo, con su conteo entre paréntesis (`KAM C ( 10 )`), y las filas siguientes la dejan vacía.

**Cómo se limpia:** se rellena cada celda vacía con el valor de la fila de arriba (`ffill`), se dejan solo las filas que tienen empresa y se quitan los `( n )`.
Al final se comprueba que haya tantos registros como dice el reporte y que cada grupo tenga los que dice su `( n )`.

In [ ]:
def leer_reporte_zoho(archivo, encabezados, columnas_grupo):
    crudo = pd.read_excel(ORIGINALES / archivo, header=None)
    esperados = int(re.search(r"Cantidad de registros : (\d+)", crudo.iloc[4, 0]).group(1))
    assert crudo.iloc[6].tolist() == encabezados, crudo.iloc[6].tolist()  # el reporte no cambió de forma

    datos = crudo.iloc[7:].copy()
    datos = datos[datos[0] != "Grand Total"]
    datos[columnas_grupo] = datos[columnas_grupo].ffill()  # cada fila hereda su grupo
    return datos, esperados


def separar_conteo(serie):
    """'KAM C ( 10 )' -> ('KAM C', 10)"""
    partes = serie.str.extract(r"^(.*?)\s*\(\s*(\d+)\s*\)\s*$")
    assert partes.notna().all().all(), serie[partes.isna().any(axis=1)]
    return partes[0].str.strip(), partes[1].astype(int)


def comprobar_conteos(df, niveles):
    """niveles: [(columna_grupo, columna_conteo), ...] del más general al más específico.
    El número entre paréntesis de cada grupo debe coincidir con las filas que quedaron en él."""
    for k, (_, conteo) in enumerate(niveles):
        grupos = df.groupby([col for col, _ in niveles[:k + 1]])
        assert (grupos.size() == grupos[conteo].first()).all(), (grupos.size(), grupos[conteo].first())

## 2. Empresas por segmento
Dos niveles de grupo: categoría de membresía (Elite / Premium) → KAM. La fecha de expiración ya viene como fecha en el Excel.
`Sin asignar (cuenta ACTHO)` se deja como `Sin asignar`, para que en Looker aparezca como una opción más junto a `KAM A`…`KAM F`.

In [ ]:
ENCABEZADOS = ["Categoría de Membresía", "Propietario de Empresa [anonimizado]",
               "Nombre de Empresa [anonimizado]", "Fecha de Expiración Afiliación"]
seg, esperados = leer_reporte_zoho("Informe por segmento Elite y Premium_anonimizado.xlsx", ENCABEZADOS, columnas_grupo=[0, 1])
seg.columns = ["categoria_membresia", "kam", "id_empresa", "fecha_expiracion"]
seg = seg[seg["id_empresa"].notna()].copy()

seg["categoria_membresia"], seg["n_categoria"] = separar_conteo(seg["categoria_membresia"])
seg["kam"], seg["n_kam"] = separar_conteo(seg["kam"])
seg["kam"] = seg["kam"].replace({"Sin asignar (cuenta ACTHO)": "Sin asignar"})
seg["id_empresa"] = seg["id_empresa"].str.strip()
seg["fecha_expiracion"] = pd.to_datetime(seg["fecha_expiracion"])
seg = seg.reset_index(drop=True)
seg.groupby(["categoria_membresia", "kam"]).size()

## 3. Validaciones
- Hay tantos registros como dice el reporte (89), y cada segmento y cada KAM dentro de su segmento tienen los que dice su `( n )`.
- Cada empresa aparece una sola vez, con formato `EMP-0000` y con fecha de expiración.

In [ ]:
assert len(seg) == esperados == 89, (len(seg), esperados)
comprobar_conteos(seg, [("categoria_membresia", "n_categoria"), ("kam", "n_kam")])
assert seg["id_empresa"].is_unique and seg["id_empresa"].str.fullmatch(r"EMP-\d{4}").all()
assert set(seg["categoria_membresia"]) == {"Elite", "Premium"}
assert seg["kam"].str.fullmatch(r"KAM [A-Z]|Sin asignar").all()
assert seg["fecha_expiracion"].notna().all()
seg = seg.drop(columns=["n_categoria", "n_kam"])

## 4. Vencimiento
Para el análisis de renovación se calcula, respecto a la fecha del reporte (13-11-2026):
- `dias_para_vencer`: negativo si ya venció.
- `estado_vencimiento`: `Vencida`, `Vence en 60 días o menos` o `Vigente`. Los 60 días son un umbral de alerta razonable para que el KAM gestione la renovación; se puede cambiar en `ALERTA_DIAS`.

Esto se calcula aquí y no en Looker para que el tablero no necesite fórmulas.

In [ ]:
FECHA_REPORTE = pd.Timestamp("2026-11-13")  # "Generado desde Zoho CRM el 13 de noviembre de 2026"
ALERTA_DIAS = 60

seg["dias_para_vencer"] = (seg["fecha_expiracion"] - FECHA_REPORTE).dt.days
seg["estado_vencimiento"] = pd.cut(seg["dias_para_vencer"], bins=[-10_000, -1, ALERTA_DIAS, 10_000],
                                   labels=["Vencida", f"Vence en {ALERTA_DIAS} días o menos", "Vigente"]).astype(str)
pd.crosstab(seg["categoria_membresia"], seg["estado_vencimiento"], margins=True)

## 5. Guardar
`utf-8-sig` para que Excel y Looker Studio lean bien las tildes. Comparte `kam` e `id_empresa` con el 07_08, el 09 y el 10.

In [ ]:
salida = LIMPIOS / "11_segmento_elite_premium_limpio.csv"
seg.to_csv(salida, index=False, encoding="utf-8-sig")

assert pd.read_csv(salida).shape == (89, 6)
salida.name